##### Extracting our ZIP file

In [5]:
import zipfile
import os

zip_path = "GoogleEartth.zip"

extract_dir = "/content/EarthData"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("ZIP Extracted Successfully!")


ZIP Extracted Successfully!


#### Importing Libraries and *Dependencies*

In [6]:
!pip install -q segmentation-models-pytorch albumentations
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image
import os
import cv2


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.1 requires langchain-core<2.0.0,>=1.2.5, which is not installed.
langgraph 1.0.9 requires langchain-core>=0.1, which is not installed.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.4.2 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.17.1 which is incompatible.
langchain-classic 1.0.1 requires langchain-text-splitters<2.0.0,>=1.1.0, but you have langchain-text-splitters 0.2.4 which is incompatible.
matplotlib 3.7.5 requires numpy<2,>=1.20, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
py

#### Defining My Change Dataset

In [7]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

class ChangeDetectionDataset(Dataset):

    def __init__(self, A_DIR, B_DIR, L_DIR, transform=None):
        self.A_DIR = A_DIR
        self.B_DIR = B_DIR
        self.L_DIR = L_DIR
        self.files = sorted([
            f for f in os.listdir(A_DIR)
            if f.endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        f = self.files[idx]

        # Loading images
        imgA = np.array(Image.open(os.path.join(self.A_DIR, f)).convert("RGB"))
        imgB = np.array(Image.open(os.path.join(self.B_DIR, f)).convert("RGB"))

        # Load mask
        mask = np.array(Image.open(os.path.join(self.L_DIR, f)))

        # Converting  mask to binary
        mask = (mask > 128).astype(np.uint8)

        # Combine before + after image → 6 channels
        image = np.concatenate([imgA, imgB], axis=2)

        # Apply augmentation
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        mask = mask.unsqueeze(0).float()

        return image, mask

#### Defining Augmentation

In [8]:
train_transform = A.Compose([
    A.RandomCrop(height=256, width=256, p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=[0.485]*6, std=[0.229]*6),
    ToTensorV2()
])

#### Loading our Change Dataset

### Making Ensembling System

In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Loading and defining Dataset

In [11]:

train_dataset = ChangeDetectionDataset(
    A_DIR="/content/EarthData/GoogleEartth/Train/T1",
    B_DIR="/content/EarthData/GoogleEartth/Train/T2",
    L_DIR="/content/EarthData/GoogleEartth/Train/Label"
)

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)


#### Making Ensembling Technique

In [12]:
encoder_list = ['resnet34', 'efficientnet-b0', 'mobilenet_v2']
# here we are choosing three different pretrained model
# which is helpful for feature extraction
models = []

for encoder_name in encoder_list:
    model = smp.Unet(  # here we are using unet model best for segmentation
        encoder_name=encoder_name,
        encoder_weights='imagenet',
        in_channels=6, # 6 channel ,since have two iamage
        classes=1,
        activation='sigmoid'
    ).to(device)

    models.append(model)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

C:\Users\rahul\anaconda3\myanaconda\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rahul\.cache\huggingface\hub\models--smp-hub--resnet34.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

C:\Users\rahul\anaconda3\myanaconda\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rahul\.cache\huggingface\hub\models--smp-hub--efficientnet-b0.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

C:\Users\rahul\anaconda3\myanaconda\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rahul\.cache\huggingface\hub\models--smp-hub--mobilenet_v2.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

#### Training Pipeline

##### Defining Training Setup

In [13]:
from torch.utils.data import DataLoader, random_split

full_dataset = ChangeDetectionDataset(
    A_DIR="/content/EarthData/GoogleEartth/Train/T1",
    B_DIR="/content/EarthData/GoogleEartth/Train/T2",
    L_DIR="/content/EarthData/GoogleEartth/Train/Label",
    transform=train_transform
)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

#### Fitting my Training Model

In [14]:
from tqdm import tqdm

criterion = nn.BCELoss()
best_loss = float("inf")

optimizers = []
for m in models:
    optimizers.append(torch.optim.Adam(m.parameters(), lr=1e-4))

for epoch in range(15):
    print(f"Starting Epoch: {epoch+1}")

    total_loss = 0
    num_batches = 0

    for images, masks in tqdm(train_loader):
        images = images.to(device)
        masks = masks.to(device)

    
        model_idx = num_batches % len(models)  
        model = models[model_idx]
        optimizer = optimizers[model_idx]

        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

Starting Epoch: 1


100%|████████████████████████████████████████████████████████████████████████████| 159/159 [01:33<00:00,  1.71it/s]


Epoch 1 | Loss: 0.6540
Starting Epoch: 2


100%|████████████████████████████████████████████████████████████████████████████| 159/159 [01:29<00:00,  1.77it/s]


Epoch 2 | Loss: 0.4325
Starting Epoch: 3


...

Epoch 13 | Loss: 0.1754
Starting Epoch: 14


100%|████████████████████████████████████████████████████████████████████████████| 159/159 [01:29<00:00,  1.77it/s]


Epoch 14 | Loss: 0.1748
Starting Epoch: 15


100%|████████████████████████████████████████████████████████████████████████████| 159/159 [01:30<00:00,  1.76it/s]

Epoch 15 | Loss: 0.1562


##### VALIDATION

In [16]:

val_loss = 0

for model in models:
    model.eval()

with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        preds = 0

        for model in models:
            preds += model(images)

        preds = preds / len(models)

        loss = criterion(preds, masks)

        val_loss += loss.item()

val_loss = val_loss / len(val_loader)

print(f"Validation Loss: {val_loss:.4f}")

Validation Loss: 0.1327


##### Saving Our Model

In [17]:
if val_loss < best_loss:
    best_loss = val_loss

    for i, model in enumerate(models):
        torch.save(model.state_dict(), f"best_model_{i}.pth")

    print("Best model saved")

Best model saved


##### Loading Test Data

In [22]:
import zipfile
import os

zip_path = "GooglEarthTest.zip"

extract_dir = "/content/BLUEEarthData"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

#### Defining Test Data

In [24]:
import numpy as np
from PIL import Image
import torch
from torchvision.transforms import functional as F

def predict_change_any_size(before_path, after_path, models, device, strong_thresh=0.25):

    before_img = np.array(Image.open(before_path).convert('RGB'))
    after_img  = np.array(Image.open(after_path).convert('RGB'))

    H, W = before_img.shape[:2]

    image = np.concatenate([before_img, after_img], axis=2).astype(np.float32)

    mean = np.array([0.485, 0.456, 0.406] * 2)
    std  = np.array([0.229, 0.224, 0.225] * 2)

    image = (image / 255.0 - mean) / std

    tensor = torch.from_numpy(image.transpose(2,0,1)).unsqueeze(0).float().to(device)

    p_final = torch.zeros((1,1,H,W)).to(device)

    with torch.no_grad():
        for model in models:
            pred = model(tensor)
            p_final += pred

    p_final /= len(models)

    change_mask = (p_final[0,0] >= strong_thresh).cpu().numpy().astype(np.uint8)

    return None, None, change_mask

#### Evaluation

In [40]:
acc_list = []
prec_list = []
rec_list = []
f1_list = []
iou_list = []

for img in sorted(os.listdir(gt_dir)):

    before = os.path.join(before_dir, img)
    after  = os.path.join(after_dir, img)
    gt_path = os.path.join(gt_dir, img)

    if not os.path.exists(before) or not os.path.exists(after):
        print("Skipping:", img)
        continue

    _, _, pred = predict_change_any_size(
        before,
        after,
        models=models,
        device=device,
        strong_thresh=0.099
    )

    gt = np.array(Image.open(gt_path).convert("L"))
    gt = (gt > 127).astype(np.uint8)

    gt = gt.flatten()
    pred = pred.flatten()

    acc_list.append(accuracy_score(gt, pred))
    prec_list.append(precision_score(gt, pred, zero_division=0))
    rec_list.append(recall_score(gt, pred, zero_division=0))
    f1_list.append(f1_score(gt, pred, zero_division=0))
    iou_list.append(jaccard_score(gt, pred, zero_division=0))

print("Accuracy :", np.mean(acc_list))
print("Precision:", np.mean(prec_list))
print("Recall   :", np.mean(rec_list))
print("F1 Score :", np.mean(f1_list))
print("IoU      :", np.mean(iou_list))

Accuracy : 0.9104646814280543
Precision: 0.18889678316916508
Recall   : 0.6428296554719737
F1 Score : 0.274589437786785
IoU      : 0.1777791871310859


### Testing visulization

#### Test 1

In [41]:
before_path = "t1A.jpeg"
after_path = "t2B.jpeg"

before_img = np.array(Image.open(before_path).convert('RGB'))
after_img = np.array(Image.open(after_path).convert('RGB'))

# 
after_img = np.array(Image.fromarray(after_img).resize((before_img.shape[1], before_img.shape[0])))

H, W = before_img.shape[:2]


test_image = np.concatenate([before_img, after_img], axis=2).astype(np.float32)

test_image = (test_image / 255.0 - 0.485) / 0.229

test_tensor = torch.from_numpy(test_image.transpose(2,0,1)).unsqueeze(0).float().to(device)

# Ensemble prediction
p_final = torch.zeros((1,1,H,W)).to(device)

with torch.no_grad():
    for model in models:
        pred = model(test_tensor)
        p_final += pred

p_final /= len(models)

strong_thresh = 0.25
change_mask = (p_final[0,0] >= strong_thresh).cpu().numpy().astype(np.uint8)

Image.fromarray((change_mask*255).astype(np.uint8)).save("change_map.png")

overlay = after_img.copy()
overlay[change_mask == 1] = [255,0,0]

Image.fromarray(overlay).save("red_highlight.png")

print("Ensemble maps saved")

Ensemble maps saved


#### Test 2

In [47]:
before_path = "S_1.png"
after_path = "S_2.png"

before_img = np.array(Image.open(before_path).convert('RGB'))
after_img = np.array(Image.open(after_path).convert('RGB'))

# 
after_img = np.array(Image.fromarray(after_img).resize((before_img.shape[1], before_img.shape[0])))

H, W = before_img.shape[:2]


test_image = np.concatenate([before_img, after_img], axis=2).astype(np.float32)

test_image = (test_image / 255.0 - 0.485) / 0.229

test_tensor = torch.from_numpy(test_image.transpose(2,0,1)).unsqueeze(0).float().to(device)

# Ensemble prediction
p_final = torch.zeros((1,1,H,W)).to(device)

with torch.no_grad():
    for model in models:
        pred = model(test_tensor)
        p_final += pred

p_final /= len(models)

strong_thresh = 0.25
change_mask = (p_final[0,0] >= strong_thresh).cpu().numpy().astype(np.uint8)

Image.fromarray((change_mask*255).astype(np.uint8)).save("change_map2.png")

overlay = after_img.copy()
overlay[change_mask == 1] = [255,0,0]

Image.fromarray(overlay).save("red_highlight2.png")

print("Ensemble2 maps saved")

Ensemble2 maps saved


#### Test 3

In [49]:
before_path = "A_35.png"
after_path = "B_35.png"

before_img = np.array(Image.open(before_path).convert('RGB'))
after_img = np.array(Image.open(after_path).convert('RGB'))

# 
after_img = np.array(Image.fromarray(after_img).resize((before_img.shape[1], before_img.shape[0])))

H, W = before_img.shape[:2]


test_image = np.concatenate([before_img, after_img], axis=2).astype(np.float32)

test_image = (test_image / 255.0 - 0.485) / 0.229

test_tensor = torch.from_numpy(test_image.transpose(2,0,1)).unsqueeze(0).float().to(device)

# Ensemble prediction
p_final = torch.zeros((1,1,H,W)).to(device)

with torch.no_grad():
    for model in models:
        pred = model(test_tensor)
        p_final += pred

p_final /= len(models)

strong_thresh = 0.10
change_mask = (p_final[0,0] >= strong_thresh).cpu().numpy().astype(np.uint8)

Image.fromarray((change_mask*255).astype(np.uint8)).save("change_map3.png")

overlay = after_img.copy()
overlay[change_mask == 1] = [255,0,0]

Image.fromarray(overlay).save("red_highlight3.png")
print("Ensemble3 maps saved")


Ensemble3 maps saved
